# 📘 09 — Pandas & Kaggle Data
*Part of the Full Stack Data Science & AI booklet. Run cells top to bottom (Shift + Enter). Install libraries first: `pip install -r ../requirements.txt`.*

In [1]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import os, sys, math, random, json, time, copy, warnings
from IPython.display import display
warnings.filterwarnings("ignore")

# 🔟 Pandas — Data analysis in tables
Pandas has two main objects: **Series** (1 column) and **DataFrame** (a table). Install: `pip install pandas`.
## 10.0 Create practice CSV datasets
We create realistic datasets in a `data/` folder. (For real ones, see the Kaggle section below.)

In [2]:
import numpy as np, pandas as pd, os
os.makedirs("data", exist_ok=True)
rng = np.random.default_rng(42)

# 1) Students
n = 60
students = pd.DataFrame({
    "student_id": range(1, n + 1),
    "name": [f"Student_{i}" for i in range(1, n + 1)],
    "gender": rng.choice(["M", "F"], n),
    "branch": rng.choice(["CSE", "ECE", "MECH", "CIVIL"], n),
    "study_hours": rng.uniform(1, 10, n).round(1),
    "attendance": rng.integers(50, 100, n),
})
students["marks"] = (25 + 6 * students.study_hours + 0.25 * students.attendance + rng.normal(0, 6, n)).clip(0, 100).round(1)
students.loc[rng.choice(n, 5, replace=False), "marks"] = np.nan          # missing values on purpose
students.to_csv("data/students.csv", index=False)

# 2) Sales
dates = pd.date_range("2025-01-01", periods=365)
sales = pd.DataFrame({
    "date": dates,
    "product": rng.choice(["Rice", "Oil", "Milk", "Bread", "Eggs"], 365),
    "city": rng.choice(["Delhi", "Pune", "Chennai"], 365),
    "units": rng.integers(1, 30, 365),
    "price": rng.choice([40, 120, 55, 35, 70], 365),
})
sales["revenue"] = sales.units * sales.price
sales.to_csv("data/sales.csv", index=False)

# 3) Houses (for regression)
m = 300
area = rng.integers(500, 3500, m); bedrooms = rng.integers(1, 6, m); age = rng.integers(0, 40, m)
dist = rng.uniform(1, 25, m).round(1)
price = 20 + 0.045 * area + 8 * bedrooms - 0.6 * age - 1.2 * dist + rng.normal(0, 10, m)   # in lakhs
pd.DataFrame({"area_sqft": area, "bedrooms": bedrooms, "age_years": age, "dist_city_km": dist,
              "location": rng.choice(["Urban", "Suburban", "Rural"], m), "price_lakhs": price.round(2)}
             ).to_csv("data/houses.csv", index=False)

# 4) Passengers (for classification / EDA, with missing values)
k = 500
pclass = rng.choice([1, 2, 3], k, p=[.25, .25, .5]); sex = rng.choice(["male", "female"], k)
ageP = rng.normal(30, 13, k).clip(1, 80).round(0)
fare = np.select([pclass == 1, pclass == 2], [90, 40], 15) + rng.exponential(10, k)
logit = -0.5 + (sex == "female") * 2.2 - (pclass - 1) * 0.9 - (ageP - 30) * 0.02
survived = (rng.random(k) < 1 / (1 + np.exp(-logit))).astype(int)
p = pd.DataFrame({"pclass": pclass, "sex": sex, "age": ageP, "fare": fare.round(2),
                  "embarked": rng.choice(["S", "C", "Q"], k, p=[.7, .2, .1]), "survived": survived})
p.loc[rng.choice(k, 60, replace=False), "age"] = np.nan
p.loc[rng.choice(k, 5, replace=False), "embarked"] = np.nan
p.to_csv("data/passengers.csv", index=False)
print(os.listdir("data"))

['students.csv', 'Investment.csv', 'car-mpg.csv', 'sales.csv', 'passengers.csv', 'Data.csv', 'houses.csv', 'tips_dataset.csv', 'Salary_Data.csv', 'student_info.csv']


#### 🔍 How it works
- `np.random.default_rng(42)` builds a random generator; `rng.choice`, `rng.uniform`, `rng.integers`, `rng.normal` create realistic columns.
- `students.loc[idx, "marks"] = np.nan` **injects missing values on purpose** so you can practise cleaning. The `price` formula includes noise (`rng.normal`) so the relation isn't perfect — like real data.
- `.to_csv(path, index=False)` saves a table (without the row numbers).

## 10.1 Series & DataFrame

In [3]:
s = pd.Series([10, 20, 30], index=["a", "b", "c"])
print(s, s["b"], s.mean(), sep="\n")

df = pd.DataFrame({"name": ["Riya", "Arjun", "Sam"], "age": [22, 19, 25], "city": ["HYD", "VZG", "HYD"]})
print(df)
print(df.shape, df.columns.tolist(), df.dtypes, sep="\n")

a    10
b    20
c    30
dtype: int64
20
20.0
    name  age city
0   Riya   22  HYD
1  Arjun   19  VZG
2    Sam   25  HYD
(3, 3)
['name', 'age', 'city']
name      str
age     int64
city      str
dtype: object


#### 🔍 How it works
- **Series** = one labelled column; **DataFrame** = a table (many Series sharing an index).
- `.shape` (rows, cols), `.columns`, `.dtypes` → the first three things to check about any DataFrame.

## 10.2 Reading CSV & first look at data
`pd.read_csv()`, `read_excel()`, `read_json()`, `read_sql()`.

In [4]:
df = pd.read_csv("data/students.csv")
print(df.head(3))          # first rows   (tail() = last rows)
df.info()                  # types + non-null counts
print(df.describe())       # count, mean, std, min, quartiles, max
print(df.isnull().sum())   # missing values per column
print(df["branch"].value_counts())
print(df.nunique())

   student_id       name gender branch  study_hours  attendance  marks
0           1  Student_1      M  CIVIL          7.0          57   87.6
1           2  Student_2      F   MECH          5.2          79    NaN
2           3  Student_3      F    ECE          6.1          54   74.1
<class 'pandas.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   student_id   60 non-null     int64  
 1   name         60 non-null     str    
 2   gender       60 non-null     str    
 3   branch       60 non-null     str    
 4   study_hours  60 non-null     float64
 5   attendance   60 non-null     int64  
 6   marks        55 non-null     float64
dtypes: float64(2), int64(2), str(3)
memory usage: 3.4 KB
       student_id  study_hours  attendance       marks
count   60.000000    60.000000   60.000000   55.000000
mean    30.500000     5.301667   72.283333   75.440000
std     17.464249     2.

#### 🔍 How it works
- `read_csv` loads the file; `head()/tail()` peek at the first/last rows; `info()` → types + non-null counts (spots missing data); `describe()` → mean, std, min, quartiles, max.
- `isnull().sum()` counts missing values per column; `value_counts()` counts each category; `nunique()` counts distinct values per column.
#### 💼 Where it's used
These 6 commands are the **first minute of every data job**. You decide cleaning steps based on what they show.

## 10.3 Selecting, filtering, sorting
- `df["col"]`, `df[["a","b"]]`
- `.loc[rows, cols]` → by **label**; `.iloc[rows, cols]` → by **position**

In [5]:
print(df["name"].head(2).tolist())
print(df[["name", "marks"]].head(3))
print(df.loc[0:2, ["name", "branch"]])
print(df.iloc[0:3, 1:4])
print(df[df.marks > 80].head(3))                                  # filter
print(df[(df.branch == "CSE") & (df.attendance > 85)].shape)      # & | ~ with brackets
print(df[df.branch.isin(["CSE", "ECE"])].shape)
print(df.query("study_hours > 8 and gender == 'F'").head(2))
print(df.sort_values("marks", ascending=False).head(3))
print(df.nlargest(3, "study_hours")[["name", "study_hours"]])

['Student_1', 'Student_2']
        name  marks
0  Student_1   87.6
1  Student_2    NaN
2  Student_3   74.1
        name branch
0  Student_1  CIVIL
1  Student_2   MECH
2  Student_3    ECE
        name gender branch
0  Student_1      M  CIVIL
1  Student_2      F   MECH
2  Student_3      F    ECE
   student_id       name gender branch  study_hours  attendance  marks
0           1  Student_1      M  CIVIL          7.0          57   87.6
3           4  Student_4      M  CIVIL          7.9          82   86.7
6           7  Student_7      M  CIVIL          6.0          87   82.4
(2, 7)
(30, 7)
    student_id        name gender branch  study_hours  attendance  marks
12          13  Student_13      F    ECE          8.7          59   86.2
39          40  Student_40      F    CSE          9.7          54   89.4
    student_id        name gender branch  study_hours  attendance  marks
43          44  Student_44      M    CSE          9.7          74  100.0
55          56  Student_56      M   MECH 

#### 🔍 How it works
- `df["col"]` → Series; `df[["a","b"]]` → DataFrame (double brackets = list of columns).
- `.loc[row_labels, cols]` selects by **label** (end included); `.iloc[row_pos, col_pos]` selects by **position** (end excluded).
- `df[df.marks > 80]` is a boolean filter; combine with `&` (and), `|` (or), `~` (not) and **put each condition in brackets**; `isin([...])` checks membership; `query("...")` writes the filter as text.
- `sort_values(col, ascending=False)`, `nlargest(n, col)` rank rows.
#### 💼 Where it's used
Slicing the table to answer business questions: "top 10 customers", "orders over ₹5000 in March".

## 10.4 Cleaning: missing data, duplicates, types, new columns

In [6]:
print("Missing before:", df.marks.isnull().sum())
df["marks"] = df["marks"].fillna(df["marks"].mean())          # fill with mean
# df = df.dropna()                                            # or drop rows
df = df.drop_duplicates()
df["result"] = np.where(df.marks >= 40, "Pass", "Fail")       # new column
df["grade"] = pd.cut(df.marks, [0, 40, 60, 75, 90, 100], labels=["F", "C", "B", "A", "A+"])
df = df.rename(columns={"study_hours": "hours"})
df["name_upper"] = df["name"].str.upper()                     # .str methods
df["attendance"] = df["attendance"].astype(float)             # change type
df["marks"] = df["marks"].apply(lambda x: min(x + 2, 100))    # apply function
df = df.drop(columns=["name_upper"])
print(df.head(3)); print("Missing after:", df.marks.isnull().sum())

Missing before: 5
   student_id       name gender branch  hours  attendance  marks result grade
0           1  Student_1      M  CIVIL    7.0        57.0  89.60   Pass     A
1           2  Student_2      F   MECH    5.2        79.0  77.44   Pass     A
2           3  Student_3      F    ECE    6.1        54.0  76.10   Pass     B
Missing after: 0


#### 🔍 How it works
- `fillna(mean)` fills gaps; `dropna()` deletes rows with gaps; `drop_duplicates()` removes repeated rows.
- `np.where(cond, "Pass", "Fail")` creates a new column from a condition; `pd.cut(values, bins, labels)` converts numbers into grade bands.
- `rename`, `astype`, `.str.upper()`, `.apply(func)` = rename columns, change types, work on text, apply any function to each value.
#### 💼 Where it's used
**Data cleaning is the biggest part of a data job.** Missing values, wrong types and duplicates break models silently — always check before modelling.

## 10.5 GroupBy, pivot, merge, dates

In [7]:
print(df.groupby("branch")["marks"].mean().round(1))
print(df.groupby(["branch", "gender"]).agg(avg_marks=("marks", "mean"), students=("marks", "count")).round(1))
print(pd.pivot_table(df, values="marks", index="branch", columns="gender", aggfunc="mean").round(1))
print(pd.crosstab(df.branch, df.result))

# Merge (like SQL JOIN)
left = pd.DataFrame({"id": [1, 2, 3], "name": ["A", "B", "C"]})
right = pd.DataFrame({"id": [1, 2, 4], "score": [90, 80, 70]})
print(pd.merge(left, right, on="id", how="inner"))
print(pd.merge(left, right, on="id", how="left"))
print(pd.concat([left, left]).reset_index(drop=True).shape)

# Dates
sales = pd.read_csv("data/sales.csv", parse_dates=["date"])
sales["month"] = sales.date.dt.month_name()
sales["weekday"] = sales.date.dt.day_name()
print(sales.groupby("month")["revenue"].sum().sort_values(ascending=False).head(3))
print(sales.set_index("date").resample("ME")["revenue"].sum().head(3))

branch
CIVIL    79.4
CSE      79.9
ECE      75.1
MECH     74.7
Name: marks, dtype: float64
               avg_marks  students
branch gender                     
CIVIL  F            76.8         7
       M            81.2        10
CSE    F            77.9         8
       M            82.6         6
ECE    F            74.1        10
       M            76.7         6
MECH   F            72.6         8
       M            78.0         5
gender     F     M
branch            
CIVIL   76.8  81.2
CSE     77.9  82.6
ECE     74.1  76.7
MECH    72.6  78.0
result  Pass
branch      
CIVIL     17
CSE       14
ECE       16
MECH      13
   id name  score
0   1    A     90
1   2    B     80
   id name  score
0   1    A   90.0
1   2    B   80.0
2   3    C    NaN
(6, 2)
month
February     37715
September    35340
August       32555
Name: revenue, dtype: int64
date
2025-01-31    21080
2025-02-28    37715
2025-03-31    27475
Freq: ME, Name: revenue, dtype: int64


#### 🔍 How it works
- `groupby("branch")["marks"].mean()` → **split** rows by branch → **apply** mean to each group → **combine** results (split-apply-combine). `.agg(name=("col","func"))` computes several stats at once.
- `pivot_table(values, index, columns, aggfunc)` = Excel pivot table. `crosstab` counts combinations.
- `pd.merge(a, b, on="id", how="inner"/"left")` = SQL JOIN; `pd.concat` stacks tables.
- Dates: `parse_dates` converts text to real dates; `.dt.month_name()` extracts parts; `.resample("ME")` groups by **month-end** (use `"M"` on older pandas).
#### 💼 Where it's used
Monthly sales reports, cohort analysis, customer segment summaries, KPI dashboards — the daily work of a data analyst.

## 🚀 Mini Project 7: Sales Analysis Report
Questions a business would ask: best product, best city, best month, average order.

In [8]:
print("Total revenue: ₹", sales.revenue.sum())
print("\nRevenue by product:\n", sales.groupby("product").revenue.sum().sort_values(ascending=False))
print("\nBest city:", sales.groupby("city").revenue.sum().idxmax())
print("Best weekday:", sales.groupby("weekday").revenue.sum().idxmax())
print("Avg order value: ₹", round(sales.revenue.mean(), 2))
top = sales.groupby(["city", "product"]).revenue.sum().unstack()
print(top)
# Save results
top.to_csv("data/sales_summary.csv")

Total revenue: ₹ 344145

Revenue by product:
 product
Eggs     78165
Milk     76495
Bread    67915
Rice     66600
Oil      54970
Name: revenue, dtype: int64

Best city: Pune
Best weekday: Tuesday
Avg order value: ₹ 942.86
product  Bread   Eggs   Milk    Oil   Rice
city                                      
Chennai  19040  31160  25355  15310  21390
Delhi    26935  20225  23975  19125  24500
Pune     21940  26780  27165  20535  20710


#### 🔍 How it works / 💼 Where it's used
- `.sum().idxmax()` → the **label** (city/product) with the highest value. `unstack()` pivots a 2-level group result into a table (city × product).
- Interpreting: a report should answer *what happened, where, when and how big* — here best product, best city, best weekday, average order value. Saving with `to_csv` hands the result to the business team.

## 10.6 📥 Getting real data from Kaggle
1. Create a free account on **kaggle.com** → Profile → *Settings → Create New API Token* → downloads `kaggle.json`.
2. Put it in `C:\Users\<you>\.kaggle\kaggle.json` (Mac/Linux: `~/.kaggle/kaggle.json`).
3. Download in code:
```python
# pip install kagglehub
import kagglehub
path = kagglehub.dataset_download("yasserh/titanic-dataset")
print(path)                       # folder that contains the CSV
```
or by CLI: `pip install kaggle` → `kaggle datasets download -d yasserh/titanic-dataset --unzip`

**Or** simply click *Download* on the dataset page and put the CSV next to your notebook.

Good beginner datasets: Titanic, House Prices, Iris, Students Performance, IPL Matches, Netflix Titles, Heart Disease, Diabetes.
Built-in practice data (needs internet the first time): `sns.load_dataset("tips")`, `"iris"`, `"titanic"`.

In [9]:
# Professional loading habits
# df = pd.read_csv("file.csv", encoding="latin1", sep=",", usecols=["a","b"], nrows=1000)
# df.to_csv("clean.csv", index=False); df.to_excel("out.xlsx", index=False)
try:
    import seaborn as sns
    tips = sns.load_dataset("tips")
    print(tips.head(3))
except Exception as e:
    print("Offline - skipping seaborn's built-in dataset:", type(e).__name__)

   total_bill   tip     sex smoker  day    time  size
0       16.99  1.01  Female     No  Sun  Dinner     2
1       10.34  1.66    Male     No  Sun  Dinner     3
2       21.01  3.50    Male     No  Sun  Dinner     3

#### 🔍 How it works
- `pd.read_csv` options: `encoding` (fix weird characters), `sep` (e.g. `;` or tab), `usecols` (load only needed columns → faster), `nrows` (preview a huge file), `parse_dates`, `dtype`.
- `to_csv`, `to_excel`, `to_json`, `to_sql` export. `try/except` makes the cell work even offline.

### 🧪 Practice (Section 10)
1. Which branch has the highest average marks? 2. Add a column `pass_fail`. 3. Find students with attendance < 60 and marks > 70. 4. Which product sold most units in Pune? 5. Fill missing `age` in passengers.csv with the median per `pclass`.